<a href="https://colab.research.google.com/github/YOUR_GITHUB_USER/YOUR_REPO/blob/main/track2_social_innovation_carecompass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤝 Track 2 · Social Innovation
## CareCompass: A Multilingual Assistance Navigator

**Coaching session · ~60 min**

### The real-world problem
Many families who qualify for financial or care support **never apply**. The reasons are rarely "they don't need it":
- Schemes are spread across many agencies, each with its own rules and jargon.
- Information is mostly in English, but many seniors are more comfortable in Mandarin, Malay, Tamil or a dialect.
- People don't know which schemes they qualify for, so they give up.
- Social workers spend hours on basic screening instead of real casework.

### The social-innovation angle
Let people describe their situation **in their own words and language**, then tell them in simple language which schemes they likely qualify for, *why*, and what to do next. A human caseworker stays in the loop for the final decision.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Understand | **Structured extraction** | Messy text in any language ➜ a clean `Profile` (age, income, household...) |
| Decide | **Rule engine in plain Python** | Transparent, auditable eligibility checks. *The AI does not decide.* |
| Explain | **Cross-lingual RAG** | Ask questions in Chinese, retrieve English scheme documents |
| Respond | **Plain-language generation** | A reply in the user's language at a primary-school reading level |

> ⚠️ **All schemes in this notebook are fictional.** For your project, use real information only from official sources (e.g. government agency websites), check it carefully, and show the source to users.

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `chromadb`: vector database for the scheme documents.
- `gradio`: the web UI for the demo.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai chromadb==1.5.9 gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also define `ask_gemini()`, a small wrapper that waits and retries when the free tier says *"429: too many requests"*. Free-tier keys only allow a few requests per minute, so every team will hit this during the hackathon.

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-2.5-flash"

def ask_gemini(contents, config=None, retries=3):
    """Call Gemini, waiting and retrying if we hit the free-tier rate limit."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"⏳ Rate limited by the free tier, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

## Step 3: The Scheme Knowledge Base
Each scheme has two parts:
- `rules`: **machine-readable** eligibility, checked by Python code.
- `doc`: **human-readable** description, searched by RAG and used to explain things.

Keeping these separate is the key design decision in this notebook.

In [ ]:
SCHEMES = [
    {
        "id": "S1", "name": "SilverCare Home Help",
        "rules": {"min_age": 65, "max_pci": 1500, "requires": ["needs_daily_living_help"]},
        "doc": "SilverCare Home Help sends trained care staff to the homes of seniors aged 65 and above who need help with "
               "daily activities like bathing, dressing or moving around. Households with monthly income per person of $1,500 "
               "or less pay only $5 per visit. Apply at any Senior Activity Centre or call the CareLine at 1800-000-0001.",
    },
    {
        "id": "S2", "name": "Healthy Seniors Clinic Card",
        "rules": {"min_age": 60, "citizenship": ["Citizen"], "max_pci": 2000},
        "doc": "The Healthy Seniors Clinic Card gives citizens aged 60 and above subsidised visits at partner GP and dental "
               "clinics, paying about $10 per GP visit. Monthly household income per person must be $2,000 or less. "
               "Apply online or at a Community Centre with your IC and latest payslips or pension statement.",
    },
    {
        "id": "S3", "name": "FamilyBoost Childcare Grant",
        "rules": {"citizenship": ["Citizen", "PR"], "max_pci": 2500, "requires": ["has_children_under_7"]},
        "doc": "FamilyBoost covers up to 80% of monthly childcare or infant-care fees for children under 7 at licensed centres. "
               "The child must be a citizen or PR, and monthly household income per person must be $2,500 or less. "
               "The childcare centre submits the application for you; just tell them you want to apply.",
    },
    {
        "id": "S4", "name": "StudentStart Bursary",
        "rules": {"max_pci": 1200, "requires": ["has_school_going_children"]},
        "doc": "StudentStart gives school-going children free school meals, free textbooks and uniforms, and a $300 yearly "
               "transport allowance. Monthly household income per person must be $1,200 or less. "
               "Ask your child's school general office for the form; schools can also apply on your behalf.",
    },
    {
        "id": "S5", "name": "JobBridge Skills Credit",
        "rules": {"min_age": 21, "max_age": 64, "employment": ["unemployed"]},
        "doc": "JobBridge helps unemployed adults aged 21 to 64 get back to work. You get $1,000 of training credit, "
               "a personal career coach, and $500 per month for up to 6 months while attending approved courses. "
               "Register at a JobBridge Career Centre; bring your IC and your last employment letter.",
    },
    {
        "id": "S6", "name": "Utilities Relief Voucher",
        "rules": {"requires_any": ["lives_in_rental_flat"], "or_max_pci": 1000},
        "doc": "Utilities Relief Vouchers give $150 every quarter off electricity and water bills. Households in public "
               "rental flats, or with monthly income per person of $1,000 or less, qualify. Vouchers are credited "
               "automatically for rental flats; other households apply online.",
    },
    {
        "id": "S7", "name": "Caregiver Respite Grant",
        "rules": {"requires": ["is_caregiver"]},
        "doc": "The Caregiver Respite Grant gives $250 per month to people caring at home for a family member who is frail, "
               "has a disability or needs long-term care. It can pay for day care, a helper or short respite stays so the "
               "caregiver can rest. Apply through a Caregiver Support Centre; a care assessment is needed.",
    },
]
print(f"Loaded {len(SCHEMES)} schemes.")

## Step 4: Understand the Person (Structured Extraction)
People don't fill in forms. They say things like *"我今年68岁，一个人住..."* or *"My pension is like 900 a month"*. Gemini reads any language and fills in our `Profile` schema.

`Optional[...]` fields matter: if the person **didn't say** their age, the model must return `null` instead of **guessing**. Guessing someone's income is the kind of mistake that hurts real people.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class Profile(BaseModel):
    age: Optional[int] = Field(description="Age in years, or null if not mentioned")
    household_size: Optional[int] = Field(description="Number of people living together including the person, or null")
    monthly_household_income: Optional[float] = Field(description="Total monthly income of the whole household in dollars, or null")
    citizenship: Literal["Citizen", "PR", "Foreigner", "Unknown"]
    employment_status: Literal["employed", "unemployed", "retired", "student", "unknown"]
    has_children_under_7: bool
    has_school_going_children: bool
    is_caregiver: bool = Field(description="Cares at home for a frail, sick or disabled family member")
    needs_daily_living_help: bool = Field(description="Struggles with bathing, dressing, eating or moving around")
    lives_in_rental_flat: bool
    language: str = Field(description="Language the person wrote in, e.g. English, Chinese, Malay, Tamil")

EXTRACT_PROMPT = """Extract the person's situation into the schema.
Only use facts the person stated or clearly implied. If something is not mentioned, use null, "Unknown"/"unknown" or false.
Never guess income or age. Someone living alone has household_size 1.

Person's message:
"""

def extract_profile(message):
    response = ask_gemini(
        EXTRACT_PROMPT + message,
        types.GenerateContentConfig(response_mime_type="application/json", response_schema=Profile),
    )
    return response.parsed

msg_en = "I'm 68 and live alone in a 1-room rental flat. My pension is about $900 a month. My knees are bad and I struggle to bathe and clean."
msg_zh = "我今年35岁，上个月失业了。家里有老婆和两个孩子，一个3岁，一个9岁在读小学。现在家里每月收入大概2800元。"

profile_en = extract_profile(msg_en)
profile_zh = extract_profile(msg_zh)
print(profile_en.model_dump_json(indent=2))
print(profile_zh.model_dump_json(indent=2))

## Step 5: Decide Eligibility with Plain Python Rules
**Coaching point:** we do *not* ask the LLM "is this person eligible?". Eligibility affects people's money and care, so it must be:
- **consistent**: the same input always gives the same answer,
- **explainable**: we can say exactly which rule passed or failed,
- **auditable**: a caseworker or regulator can check the logic.

LLMs are good at *understanding* messy language. Code is good at *applying rules*. Use each for what it does best.

Each scheme gets one of three results: ✅ likely eligible, ❓ need more info, or ❌ not eligible.

In [ ]:
def per_capita_income(p):
    if p.monthly_household_income is None or not p.household_size:
        return None
    return p.monthly_household_income / p.household_size

def check_scheme(scheme, p):
    r = scheme["rules"]
    passed, failed, missing = [], [], []
    pci = per_capita_income(p)

    if "min_age" in r or "max_age" in r:
        if p.age is None:
            missing.append("age")
        elif p.age < r.get("min_age", 0) or p.age > r.get("max_age", 200):
            failed.append(f"age {p.age} outside {r.get('min_age', 0)}–{r.get('max_age', '∞')}")
        else:
            passed.append(f"age {p.age} ✓")

    if "citizenship" in r:
        if p.citizenship == "Unknown":
            missing.append("citizenship")
        elif p.citizenship in r["citizenship"]:
            passed.append(f"{p.citizenship} ✓")
        else:
            failed.append(f"must be {' or '.join(r['citizenship'])}")

    if "employment" in r:
        if p.employment_status == "unknown":
            missing.append("employment status")
        elif p.employment_status in r["employment"]:
            passed.append(f"{p.employment_status} ✓")
        else:
            failed.append(f"must be {' or '.join(r['employment'])}")

    for flag in r.get("requires", []):
        label = flag.replace("_", " ")
        if getattr(p, flag):
            passed.append(f"{label} ✓")
        else:
            failed.append(f"{label}: not mentioned")

    if "max_pci" in r:
        if pci is None:
            missing.append("household income and size")
        elif pci <= r["max_pci"]:
            passed.append(f"income per person ${pci:,.0f} ≤ ${r['max_pci']:,}")
        else:
            failed.append(f"income per person ${pci:,.0f} > ${r['max_pci']:,}")

    # "Either/or" rule: e.g. lives in a rental flat OR has low income
    if "requires_any" in r:
        any_flag = any(getattr(p, f) for f in r["requires_any"])
        low_income = pci is not None and pci <= r.get("or_max_pci", -1)
        if any_flag or low_income:
            passed.append("rental flat ✓" if any_flag else f"income per person ${pci:,.0f} ✓")
        elif pci is None:
            missing.append("household income and size")
        else:
            failed.append("not in rental flat and income above limit")

    if failed:
        status, why = "❌ Not eligible", "; ".join(failed)
    elif missing:
        status, why = "❓ Need more info", "missing: " + ", ".join(missing)
    else:
        status, why = "✅ Likely eligible", "; ".join(passed)
    return {"scheme_id": scheme["id"], "scheme": scheme["name"], "status": status, "why": why}

def check_all(p):
    return [check_scheme(s, p) for s in SCHEMES]

import pandas as pd
print("👵 English example (68, alone, rental flat):")
display(pd.DataFrame(check_all(profile_en)))
print("👨‍👩‍👧‍👦 Chinese example (35, unemployed, family of 4):")
display(pd.DataFrame(check_all(profile_zh)))

## Step 6: Cross-Lingual RAG for Follow-up Questions
After seeing their results, people ask things like *"老人家护理要怎么申请？"* ("How do I apply for elderly home care?").

Gemini's embedding model is **multilingual**: a question in Chinese lands close to the matching English document in vector space, so we don't need to translate our knowledge base into every language.

In [ ]:
import chromadb

chroma_client = chromadb.Client()
COLLECTION_NAME = "carecompass_schemes"
if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(COLLECTION_NAME)
collection = chroma_client.create_collection(name=COLLECTION_NAME)

def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return result.embeddings[0].values

scheme_texts = [f"{s['name']}: {s['doc']}" for s in SCHEMES]
collection.add(
    ids=[s["id"] for s in SCHEMES],
    documents=scheme_texts,
    embeddings=[get_embedding(t) for t in scheme_texts],
)

def search_schemes(question, n=2):
    hits = collection.query(query_embeddings=[get_embedding(question, "RETRIEVAL_QUERY")], n_results=n)
    return hits["documents"][0]

for q in ["老人家护理要怎么申请？", "Saya hilang kerja, ada bantuan latihan?"]:
    print(f"❓ {q}\n   ➜ top match: {search_schemes(q, n=1)[0][:90]}...\n")

## Step 7: Reply in Simple Words, in the Person's Language
The final reply combines:
- the **rule-engine results**, which are the facts (the model may not change them),
- the **scheme documents**, for how to apply, and
- **style rules**: their language, short sentences, no jargon, and a clear next step.

In [ ]:
REPLY_PROMPT = """You are CareCompass, a warm and patient assistant helping people find support schemes.
Write the reply in {language}. Use very simple words and short sentences (primary-school reading level).

Eligibility results (decided by official rules, do NOT change them):
{results}

Scheme information (only use these facts):
{docs}

Person's message: {message}

Your reply must:
1. Start with one kind sentence acknowledging their situation.
2. List the schemes marked "Likely eligible", each with one line on what it gives and one line on how to apply.
3. For "Need more info" schemes, ask ONE simple question for the missing information.
4. Do not mention schemes marked "Not eligible".
5. End with: a social worker can help them apply, and these results are a guide, not a final decision."""

def carecompass(message):
    p = extract_profile(message)
    results = check_all(p)
    relevant_ids = {r["scheme_id"] for r in results if not r["status"].startswith("❌")}
    docs = "\n".join(f"- {s['name']}: {s['doc']}" for s in SCHEMES if s["id"] in relevant_ids)
    prompt = REPLY_PROMPT.format(
        language=p.language,
        results="\n".join(f"- {r['scheme']}: {r['status']} ({r['why']})" for r in results),
        docs=docs or "(none)",
        message=message,
    )
    reply = ask_gemini(prompt).text
    return p, results, reply

_, _, reply = carecompass(msg_zh)
print(reply)

### 🤔 Coaching discussion (5 min)
- Try a message that **leaves out income**. Do you get ❓ and a follow-up question instead of a wrong ✅?
- Try Malay, Tamil or Singlish. Where does extraction break?
- Who is harmed if the system says ✅ but the person is actually not eligible? What about ❌ when they *are* eligible? Which mistake is worse here?
- Why do we **show the reason** for each result instead of just the verdict?

## Step 8: Build the Demo UI with Gradio 🏆

In [ ]:
import gradio as gr

def ui_check(message):
    if not message.strip():
        return {}, [], "Please tell me a little about your situation."
    try:
        p, results, reply = carecompass(message)
    except Exception as e:
        return {}, [], f"⚠️ Sorry, something went wrong: {e}"
    table = [[r["scheme"], r["status"], r["why"]] for r in results]
    return p.model_dump(), table, reply

def ui_ask(question):
    if not question.strip():
        return ""
    try:
        docs = "\n".join(search_schemes(question))
        prompt = (f"Answer in the same language as the question, in simple words. Use ONLY this information; "
                  f"if the answer isn't there, say a social worker can help.\n\nInformation:\n{docs}\n\nQuestion: {question}")
        return ask_gemini(prompt).text
    except Exception as e:
        return f"⚠️ Sorry, something went wrong: {e}"

with gr.Blocks(title="CareCompass") as demo:
    gr.Markdown("# 🧭 CareCompass\nTell us about your situation in any language. We'll point you to support you may qualify for.\n"
                "*Demo with fictional schemes. Results are a guide, not a final decision.*")
    with gr.Tab("Check my support"):
        msg_in = gr.Textbox(label="Your situation", lines=4)
        gr.Examples([msg_en, msg_zh, "Saya berumur 45, menjaga ibu saya yang lumpuh di rumah. Pendapatan keluarga $3,000 sebulan untuk 3 orang."], inputs=msg_in)
        check_btn = gr.Button("Find my support", variant="primary")
        reply_out = gr.Markdown()
        with gr.Accordion("For caseworkers: how we decided", open=False):
            table_out = gr.Dataframe(headers=["Scheme", "Result", "Why"], wrap=True)
            profile_out = gr.JSON(label="Extracted profile")
        check_btn.click(ui_check, inputs=msg_in, outputs=[profile_out, table_out, reply_out])
    with gr.Tab("Ask a question"):
        q_in = gr.Textbox(label="Your question (any language)", placeholder="e.g. 老人家护理要怎么申请？")
        ask_btn = gr.Button("Ask")
        a_out = gr.Markdown()
        ask_btn.click(ui_ask, inputs=q_in, outputs=a_out)

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🚀 Make It Your Own: Hackathon Extensions
- **Voice in, voice out:** many seniors find voice easier than typing. Gemini understands audio; add text-to-speech for replies.
- **Ask, don't guess:** turn it into a chat that keeps asking for missing fields until every scheme is ✅ or ❌.
- **Caseworker handoff:** a "Send to social worker" button that creates a pre-filled case summary.
- **Same pattern, other causes:** migrant-worker rights, disability services, mental-health resources, food-bank finder, legal-aid triage.

## 🎤 Pitching to Judges
- Open with **one person's story** (*"Madam Tan, 78, didn't know she qualified for three schemes"*), then show it working for her.
- Demo **two languages** live.
- Show the caseworker panel: *"AI understands, rules decide, humans approve."*

## 🛡️ Responsible AI Checklist
- [ ] Eligibility decided by transparent rules, not the LLM.
- [ ] The system asks instead of guessing when information is missing.
- [ ] Personal data: only collect what you need, don't store chats, and follow data-protection law (in Singapore, the PDPA).
- [ ] The free Gemini API tier may use your inputs to improve Google's products, so **never test with real people's data**.
- [ ] Clear disclaimer and a route to a human.